In [ ]:
# Consolidated imports (all imports moved here)

# Use https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html to enable ROCm on Windows if you have AMD GPU
# https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/howto_windows.html


import os
import json
import base64
from io import BytesIO
from pathlib import Path
from PIL import Image
from urllib.parse import urlparse, unquote
from pathlib import Path
import traceback

# Optional/extra libraries: import safely so notebook doesn't error on missing packages
try:
    from doctr.io import DocumentFile
    from doctr.models import ocr_predictor
except Exception:
    DocumentFile = None
    ocr_predictor = None

try:
    from langchain_ollama import ChatOllama
except Exception:
    ChatOllama = None

try:
    from pdf2image import convert_from_path
except Exception:
    convert_from_path = None



PDF_DIR = "C:\\Users\\senth\\OneDrive\\Receipts"
# Ensure PDF_DIR is defined (string or Path)
pdf_dir = Path(PDF_DIR)
# Save outputs to user's Downloads directory (subfolder 'ocr_outputs')
downloads_dir = Path.home() / "Downloads" / "ocr_outputs"
downloads_dir.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: {downloads_dir}")

if not pdf_dir.exists() or not pdf_dir.is_dir():
    raise FileNotFoundError(f"PDF_DIR does not exist or is not a directory: {pdf_dir}")

# Non-recursive (only top-level .pdf files)
pdf_files = [str(p) for p in pdf_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"]

# Recursive (all .pdf files under PDF_DIR)
pdf_files_recursive = [str(p) for p in pdf_dir.rglob("*.pdf") if p.is_file()]

# If you want only filenames (no directories)
pdf_file_names = [Path(p).name for p in pdf_files_recursive]

print(f"Found {len(pdf_files_recursive)} PDF(s)")

c:\src\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Outputs will be saved to: C:\Users\senth\Downloads\ocr_outputs
Found 58 PDF(s)


In [ ]:
import torch
from doctr.models.predictor import OCRPredictor
# Check GPU availability and initialize predictor via get_doctr_predictor
print(f"Is GPU available: {torch.cuda.is_available()}")
try:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
except Exception:
    pass
# Use the cached initializer to get a predictor on the best device
model = ocr_predictor (pretrained=True)
print(f"docTR predictor successfully initialized.")


In [ ]:
# OCR function using docTR
def ocr_pdf_doctr(pdf_path: str,
                  predictor=None,
                 ) -> tuple:
  
    if DocumentFile is None :
        raise ImportError("docTR (doctr) is not available. Install 'python-doctr' to use this function.")
    if predictor is None:
        try:
            predictor = ocr_predictor(det_arch =  "db_resnet50",reco_arch = "crnn_vgg16_bn", pretrained = True, assume_straight_pages=True).cuda()
        except Exception as e:
            raise ValueError("Failed to initialize docTR predictor:", e)
            return
        

    # Load the PDF as a docTR Document
    doc = DocumentFile.from_pdf(pdf_path)
   # Run inference (docTR handles batching and multi-page PDF)
    result = predictor(doc)

    # Export as JSON-like nested dict and build plain text
    json_out = result.export()

    pages_text = []
    for page in json_out.get("pages", []):
        page_lines = []
        for block in page.get("blocks", []):
            for line in block.get("lines", []):
                words = [w.get("value", "") for w in line.get("words", [])]
                if words:
                    page_lines.append(" ".join(words))
        pages_text.append("\n".join(page_lines))

    full_text = "\n\n".join([f"--- Page {i+1} ---\n" + p for i, p in enumerate(pages_text)])
    # Return JSON-exported structure for easier downstream processing and the plain text
    return json_out, full_text

print(f"Starting OCR for {len(pdf_files_recursive)} PDF(s)")
processed = 0
errors = []
model = ocr_predictor(det_arch =  "db_resnet50",reco_arch = "crnn_vgg16_bn", pretrained = True, assume_straight_pages=True).cuda()
for pdf_path in pdf_files_recursive:
    print("\n" + "="*60)
    print(f"Processing: {pdf_path}")
    try:
        res_json, plain = ocr_pdf_doctr(pdf_path, predictor=None)
        print(f"Pages: {len(res_json.get('pages', []))} | Text length: {len(plain)} chars")
        # Save outputs to downloads directory with same filename prefix
        stem = Path(pdf_path).stem
        out_txt = downloads_dir / f"{stem}.doctr.txt"
        out_json = downloads_dir / f"{stem}.doctr.json"
        with open(out_txt, "w", encoding="utf-8") as f:
            f.write(plain)
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(res_json, f, ensure_ascii=False, indent=2)
        print(f"Saved plain text -> {out_txt}")
        print(f"Saved JSON -> {out_json}")

        # Convert PDF pages to images at 200dpi and save them to downloads_dir
        images = []
        # Preferred: pdf2image at 200 DPI
        if convert_from_path is not None:
            try:
                images = convert_from_path(pdf_path, dpi=200)
                print(f"Converted PDF to {len(images)} image(s) with pdf2image at 200dpi")
            except Exception as ex:
                print("pdf2image conversion failed:", ex)
                traceback.print_exc()
                images = []
        else:
            print("pdf2image not available; will try PyMuPDF fallback")

        # Save images with same filename prefix into downloads_dir
        if images:
            for idx, im in enumerate(images, start=1):
                out_img = downloads_dir / f"{stem}.page-{idx}.png"
                try:
                    im.save(out_img)
                    print(f"Saved page image -> {out_img}")
                except Exception as s_ex:
                    print(f"Failed saving image {out_img}:", s_ex)
                    traceback.print_exc()
        else:
            print("No page images generated for this file.")

        processed += 1
    except Exception as e:
        print("Error during OCR for file:", pdf_path)
        traceback.print_exc()
        errors.append((pdf_path, repr(e)))

print("\n--- DONE ---")
print(f"Successfully processed {processed}/{len(pdf_files_recursive)} PDFs")
if errors:
    print("Errors encountered for the following files:")
    for p, err in errors:
        print(p, err)

In [ ]:
# Cell 1: Setup and Configuration
# (imports moved to the first cell)
# Initialize the Local LLM via Ollama
# Ensure Ollama is running in the background!
if ChatOllama is None:
    print("Skipping ChatOllama demo: 'langchain_ollama' not installed.")
else:
    try:
        demo = ChatOllama(model="qwen3:8b", validate_model_on_init=True, temperature=0)

        resp = demo.invoke("Please reply with just the single word: YES")
        text = getattr(resp, "content", None) or getattr(resp, "text", None) or str(resp)
        print("Ping response:", text)

        print("\nStreaming test:")
        for chunk in demo.stream("Say YES"):
            # chunk may be AIMessageChunk-like; print text field
            print(chunk.text, end="")

    except Exception as e:
        print("ChatOllama demo failed:", e)

In [2]:
# New cell: send OCR JSON + page images to qwen3-vl via Ollama
# (imports moved to the first cell)
# Get PDF file from notebook variables
PDF_TESTFILE = "R1123-1000"
IMAGES_DIR = downloads_dir  # Use the same downloads_dir as before
pdf_json_file = IMAGES_DIR / f"{PDF_TESTFILE}.doctr.json"
if not pdf_json_file.exists():
    raise FileNotFoundError(f"OCR JSON file not found: {pdf_json_file}")    
pdf_image_file = IMAGES_DIR / f"{PDF_TESTFILE}.page-1.png"
if not pdf_image_file.exists():
    raise FileNotFoundError(f"PDF page image file not found: {pdf_image_file}")    
pdf_text_file = IMAGES_DIR / f"{PDF_TESTFILE}.doctr.txt"
if not pdf_text_file.exists():
    raise FileNotFoundError(f"OCR plain text file not found: {pdf_text_file}")  


In [ ]:
# Load OCR JSON + image, compact/truncate JSON, encode image to base64 data URI, send to Ollama
import json
import base64
from pathlib import Path
from io import BytesIO
from PIL import Image

# Files expected to exist earlier in the notebook:
# pdf_json_file, pdf_image_file, ollama
pdf_json_file = Path(pdf_json_file)
pdf_image_file = Path(pdf_image_file)

# Basic checks
if not pdf_json_file.exists():
    raise FileNotFoundError(f"OCR JSON not found: {pdf_json_file}")
if not pdf_image_file.exists():
    raise FileNotFoundError(f"Image file not found: {pdf_image_file}")

# Load full JSON
with open(pdf_json_file, "r", encoding="utf-8") as f:
    full_json = json.load(f)

def compact_doctr_json(full_json, max_chars=20000, max_words_per_page=200):
    """
    Return a compact JSON containing per-page text (trimmed) and up to max_words_per_page tokens (value+confidence).
    If the resulting serialized JSON is > max_chars, it reduces words/page and text length and marks 'truncated'.
    """
    pages = []
    for i, page in enumerate(full_json.get("pages", [])):
        # assemble page text (short)
        lines = []
        words = []
        for block in page.get("blocks", []):
            for line in block.get("lines", []):
                vals = [w.get("value", "") for w in line.get("words", []) if w.get("value")]
                if vals:
                    lines.append(" ".join(vals))
                for w in line.get("words", []):
                    v = w.get("value", "")
                    c = w.get("confidence", None)
                    if v:
                        words.append({"v": v, "c": round(c, 3) if c is not None else None})
        pages.append({
            "page": i + 1,
            "text": "\n".join(lines)[:1000],          # limit text per page
            "words": words[:max_words_per_page]     # limit token count per page
        })

    compact = {"pages": pages}
    s = json.dumps(compact, ensure_ascii=False, separators=(",", ":"))
    if len(s) <= max_chars:
        return compact

    # If still too big, reduce further by lowering words-per-page and text length
    # Heuristic: reduce words and text, then mark as truncated
    reduced_words = max(10, int(max_words_per_page * max_chars / max(len(s), 1) / 4))
    pages_reduced = []
    for p in pages:
        pages_reduced.append({
            "page": p["page"],
            "text": p["text"][:300],
            "words": p["words"][:reduced_words]
        })
    return {"pages": pages_reduced, "truncated": True}

# Create compact JSON (adjust max_chars if needed)
compact_json = "" #compact_doctr_json(full_json, max_chars=20000, max_words_per_page=200)
compact_str = " " #json.dumps(compact_json, ensure_ascii=False, indent=2)

# Read/resize/compress image to keep prompt small (converts to JPEG)
img = Image.open(pdf_image_file)

max_size = 256
width, height = img.size
    
if width <= max_size and height <= max_size:
        new_height = height
        new_width = width
        img_resized = img
else:
    ratio = min(max_size / width, max_size / height)
    new_width = int(width * ratio)
    new_height = int(height * ratio)
    img_resized = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

print(f"Original image size: {width}x{height}, resized to: {new_width}x{new_height}")
img = img_resized
#max_w = 1024
#if img.width > max_w:
#   ratio = max_w / img.width
#  img = img.resize((max_w, int(img.height * ratio)), Image.LANCZOS)

buf = BytesIO()
img.save(buf, format="PNG", quality=95)
img_bytes = buf.getvalue()
img_b64 = base64.b64encode(img_bytes).decode("ascii")
data_uri = f"data:image/png;base64,{img_b64}"



instruction = ( 
    "You are an assistant."
    "Return: 1) Consolidated plain-text extraction (one paragraph per page), 2) Receipt Number, Date and Total Amount. "
    "Do NOT hallucinate text not present in the JSON/image."
)
prompt_parts = [
    instruction,
    "\n\n--- PAGE 1 IMAGE ---",
    f"![page1]({data_uri})"
]
prompt = "\n".join(prompt_parts)

# Quick sanity size check
size_est = len(prompt)
print (f"Prompt size: ~{size_est:,} chars")
if size_est > 150_000:
    print(f"Warning: prompt size ~{size_est:,} chars (consider lowering max_chars or using a hosted image).")

# Initialize Ollama qwen3-vl
try:
    if ChatOllama is None:
        raise RuntimeError("ChatOllama is not available")
    ollama = ChatOllama(model="qwen3-vl", validate_model_on_init=True, 
    #num_ctx: Total context window (Input + Output)
    num_ctx=132768, 
    # num_predict: Max tokens for a single response (-1 for infinite/max possible)
    num_predict=4096, 
    
    temperature=0)
    print("Initialized ChatOllama with model qwen3-vl")
except Exception as e:
    print("Failed to initialize Ollama model:", e)
    ollama = None
    
# Send to model
if ollama is None:
    print("Ollama client `ollama` is not initialized. Initialize `ollama` first.")
else:
    try:
        resp = ollama.invoke(prompt)
        text = getattr(resp, "content", None) or getattr(resp, "text", None) or str(resp)
        print("\n--- MODEL RESPONSE ---")
        print(text)
    except Exception as e:
        print("Error invoking model:", e)

In [ ]:
# Initialize Ollama qwen3-vl
try:
    if ChatOllama is None:
        raise RuntimeError("ChatOllama is not available")
    ollama = ChatOllama(model="qwen3-vl", validate_model_on_init=True, temperature=0)
    print("Initialized ChatOllama with model qwen3-vl")
except Exception as e:
    print("Failed to initialize Ollama model:", e)
    ollama = None

# Instruction: treat confidence >= 0.9 as HIGH_CONF
instruction = (
    "You are an assistant. Below is OCR JSON produced by docTR and the corresponding page images. "
    "Treat any word/token that has a 'confidence' value >= 0.7 as HIGH_CONF and accept it as reliable. "
    "For words with lower confidence, mark them with [LOW_CONF] in your output. "
    "Return: 1) A consolidated plain-text extraction (one paragraph per page), 2) Receipt Number, Date and Total Amount found on the receipt. "
    "Do NOT hallucinate text not present in the JSON or images."
)

# Build prompt (include JSON and embedded images)
prompt_parts = [instruction, "\n\nDOCTR_JSON:", json.dumps(res_json, ensure_ascii=False)]
for i, uri in enumerate(data_uris, start=1):
    prompt_parts.append(f"\n\n--- PAGE {i} IMAGE ---\n![page{i}]({uri})")

prompt = "\n".join(prompt_parts)
print ('\n--- PROMPT PREVIEW ---')
print(prompt)
# Send prompt to model and print response
if ollama is not None:
    try:
        resp = ollama.invoke(prompt)
        text = getattr(resp, "content", None) or getattr(resp, "text", None) or str(resp)
        print('\n--- MODEL RESPONSE ---')
        print(text)
    except Exception as e:
        print("Error invoking model:", e)
else:
    print("Model not available; skipping inference.")

In [ ]:
import traceback
def check(name, fn):
    try:
        fn()
        print(name, "OK")
    except Exception as e:
        print(name, "ERROR:", e)
        traceback.print_exc()

check("pdf2image", lambda: __import__("pdf2image"))
check("PyMuPDF (fitz)", lambda: __import__("fitz"))
check("pdfminer.six", lambda: __import__("pdfminer"))

In [ ]:
import ollama
from pathlib import Path
import json
pdf_json_file = Path(pdf_json_file)
if not pdf_json_file.exists():
    raise FileNotFoundError(f"OCR JSON file not found: {pdf_json_file}")
with open(pdf_json_file, "r", encoding="utf-8") as f:
    full_json = json.load(f)

#compact_json = compact_doctr_json(full_json, max_chars=20000, max_words_per_page=200)
#compact_str = json.dumps(compact_json, ensure_ascii=False) 
#ocr_prompt = "Transcribe all text from this image exactly as it appears. Maintain formatting where possible."

instruction = (
    "You are an assistant. Below is docTR OCR output (JSON). "
    "For any word whose 'confidence' < 0.7, mark it in the consolidated text as [LOW_CONF]word. "
    "Return EXACTLY one valid JSON object (no extra text) with keys: "
    "'pages' (list of {page:int, text:str}), 'receipt_number' (str|null), "
    "'date' (ISO YYYY-MM-DD or null), 'total_amount' (str|null). "
    "Do NOT hallucinate. If a field isn't present, return null for it."
)
ocr_prompt = instruction #+ "\n\nDOCTR_JSON:\n" + compact_str
print (ocr_prompt)
print (pdf_image_file)

# Re-enabled: call ollama.chat to send the prompt and print the model response.
response = ollama.chat(
    model='qwen3-vl',
    messages=[{
        'role': 'user',
        'content': ocr_prompt,
        'images' : [pdf_image_file ]

    }],
    
    options={
        'num_ctx': 32768,      # Large context to avoid truncation
        'temperature': 0       # Set to 0 for consistent OCR accuracy
    }
)
print(response['message']['content'])

# Alternative (example) using the invoke-style API:
# # resp = ollama.invoke(ocr_prompt)
# # print(getattr(resp, 'content', getattr(resp, 'text', str(resp))))

You are an assistant. Below is docTR OCR output (JSON). For any word whose 'confidence' < 0.7, mark it in the consolidated text as [LOW_CONF]word. Return EXACTLY one valid JSON object (no extra text) with keys: 'pages' (list of {page:int, text:str}), 'receipt_number' (str|null), 'date' (ISO YYYY-MM-DD or null), 'total_amount' (str|null). Do NOT hallucinate. If a field isn't present, return null for it.
C:\Users\senth\Downloads\ocr_outputs\R1123-1000.page-1.png
